# Exam management — dummy dataset generator

Build order matters here. The **inputs** are the small tables a human would edit;
`registrations` is **derived** from them at the end, never authored directly.

| Table | Kind | Rows |
|---|---|---|
| `students`, `sections`, `courses`, `rooms`, `slots` | input | entity |
| `cohort_cores` | input | which cores each cohort takes |
| `student_electives` | input | **one row per student** — the only per-student choice |
| `teaching_assignments` | input | teacher to course-group |
| `registrations` | **derived** | cohort cores x students + electives |
| `seats`, `stripes`, `stripe_adjacency` | **derived** | from room geometry |

Edit any input cell and re-run from there.

## 0. Config

Everything tunable lives here.

In [ ]:
import os, random, itertools, datetime as dt
import pandas as pd

SEED = 20260313
OUT  = "data"

# --- cohorts: (id, program, year, n_sections, section_size) ---
COHORTS = [
    ("BT2", "BTECH", 2, 9, 50),
    ("BT3", "BTECH", 3, 9, 50),
    ("BB2", "BBA",   2, 3, 50),
    ("BB3", "BBA",   3, 3, 50),
]

MAX_SECTIONS_PER_TEACHER_PER_YEAR = 3
ELECTIVE_GROUP_SIZE = 50

# --- rooms: (prefix, bench_width, seats_per_bench, cols, rows, count) ---
ROOM_TYPES = [
    ("S", 2, 1, 3, 6, 20),   # 2-wide bench, 1 student
    ("M", 3, 2, 3, 7, 20),   # 3-wide bench, 2 students
]

EXAM_START   = dt.date(2026, 3, 2)
EXAM_END     = dt.date(2026, 3, 27)
WEEKDAYS_ONLY = True
SESSIONS     = ("AM", "PM")

random.seed(SEED)
os.makedirs(f"{OUT}/sections", exist_ok=True)

## 1. Students and sections

In [ ]:
LETTERS = "ABCDEFGHIJKLMNOP"

_stu, _sec = [], []
for cid, prog, yr, nsec, size in COHORTS:
    seq = 1
    for s in range(nsec):
        sec_id = f"{cid}_{LETTERS[s]}"
        for _ in range(size):
            _stu.append(dict(student_id=f"{cid}S{seq:04d}", cohort_id=cid,
                             section_id=sec_id, roll_no=seq))
            seq += 1
        _sec.append(dict(section_id=sec_id, cohort_id=cid, program=prog,
                         year=yr, label=LETTERS[s], strength=size))

students = pd.DataFrame(_stu)
sections = pd.DataFrame(_sec)
print(len(students), "students in", len(sections), "sections")
students.head()

## 2. Courses

`course_id` is the offering; `paper_id` is the canonical paper. They are 1:1 in this dataset — set two offerings to the same `paper_id` to test the shared-paper case.

In [ ]:
CORES = {
 "BT2": ["DSA","OOPS","LA","DAP","OS","DLD"],
 "BT3": ["ML","AI","SE","COA","IOT","CS_SKILL"],
 "BB2": ["MACROECON","BUSSTAT","BUSCOMM"],
 "BB3": ["BUSLAW","STRATMGMT","OPSMGMT","HAPPINESS"],
}
# (code, target enrolment) -- deliberately lopsided
ELECTIVES = {
 "BT2": [("SEMIPHY",110), ("QM",90),        ("DISCMATH",250)],
 "BT3": [("BIGDATA",115), ("CYBERSEC",95),  ("ADVDS",240)],
 "BB2": [("FINANCE",55),  ("MARKETING",55), ("DESTHINK",40)],
 "BB3": [("BIM",50),      ("ETHICS",50),    ("ORGBEH",50)],
}
TITLES = {
 "DSA":"Data Structures and Algorithms","OOPS":"Object Oriented Programming",
 "LA":"Linear Algebra","DAP":"Data Analysis in Python","OS":"Operating Systems",
 "DLD":"Digital Logic Design","ML":"Machine Learning","AI":"Artificial Intelligence",
 "SE":"Software Engineering","COA":"Computer Architecture and Organization",
 "IOT":"Internet of Things","CS_SKILL":"Coding Skills","MACROECON":"Macroeconomics",
 "BUSSTAT":"Business Statistics","BUSCOMM":"Business Communication",
 "BUSLAW":"Business Law","STRATMGMT":"Strategic Management",
 "OPSMGMT":"Operations Management","HAPPINESS":"Happiness",
 "SEMIPHY":"Semiconductor Physics","QM":"Quantum Mechanics",
 "DISCMATH":"Discrete Mathematics","BIGDATA":"Big Data Analysis",
 "CYBERSEC":"Cyber Security","ADVDS":"Advanced Data Science","FINANCE":"Finance",
 "MARKETING":"Marketing","DESTHINK":"Design Thinking",
 "BIM":"Bringing Ideas to Market","ETHICS":"Ethics and Integrity in Business",
 "ORGBEH":"Organizational Behaviour",
}
DEPT = {
 "DSA":"CSE","OOPS":"CSE","DAP":"CSE","OS":"CSE","ML":"CSE","AI":"CSE","SE":"CSE",
 "CS_SKILL":"CSE","BIGDATA":"CSE","CYBERSEC":"CSE","ADVDS":"CSE",
 "COA":"ECE","IOT":"ECE","DLD":"ECE","LA":"MATH","DISCMATH":"MATH",
 "QM":"PHY","SEMIPHY":"PHY","MACROECON":"MGMT","BUSSTAT":"MGMT","BUSCOMM":"MGMT",
 "STRATMGMT":"MGMT","OPSMGMT":"MGMT","FINANCE":"MGMT","MARKETING":"MGMT","BIM":"MGMT",
 "BUSLAW":"LAW","ETHICS":"LAW","HAPPINESS":"HUM","DESTHINK":"HUM","ORGBEH":"HUM",
}

_c = []
for cid, *_ in COHORTS:
    for code_ in CORES[cid]:
        _c.append((f"{cid}_{code_}", f"{cid}_{code_}", f"{cid}-{code_}",
                   TITLES[code_], cid, "CORE", DEPT[code_]))
    for code_, _n in ELECTIVES[cid]:
        _c.append((f"{cid}_{code_}", f"{cid}_{code_}", f"{cid}-{code_}",
                   TITLES[code_], cid, "ELECTIVE", DEPT[code_]))

courses = pd.DataFrame(_c, columns=["course_id","paper_id","code","title",
                                    "cohort_id","kind","dept"])
courses.kind.value_counts()

## 3. Input — `cohort_cores`

19 rows. Which cores every student of a cohort takes.

In [ ]:
cohort_cores = (courses[courses.kind == "CORE"][["cohort_id", "course_id"]]
                .sort_values(["cohort_id", "course_id"]).reset_index(drop=True))
cohort_cores

## 4. Input — `student_electives`

**One row per student.** This is the table you edit to change who takes what.

In [ ]:
_rows = []
for cid, *_ in COHORTS:
    pool = students.loc[students.cohort_id == cid, "student_id"].tolist()
    random.shuffle(pool)
    i = 0
    for code_, n in ELECTIVES[cid]:
        for s in sorted(pool[i:i+n]):
            _rows.append((s, f"{cid}_{code_}"))
        i += n
    assert i == len(pool), f"{cid}: elective targets sum to {i}, cohort has {len(pool)}"

student_electives = (students[["student_id","cohort_id","section_id","roll_no"]]
                     .merge(pd.DataFrame(_rows, columns=["student_id","elective_id"]),
                            on="student_id")
                     .sort_values("student_id").reset_index(drop=True))

assert len(student_electives) == len(students), "every student needs exactly one elective"
print(len(student_electives), "rows — one per student")
student_electives.head()

## 5. Derived — `registrations`

One function. Cores expand from cohort, electives join in. Long format because everything downstream (conflict graph, seating, duty exclusion) is a join on this shape.

In [ ]:
def build_registrations(student_electives, cohort_cores):
    core = (student_electives[["student_id", "cohort_id"]]
            .merge(cohort_cores, on="cohort_id")[["student_id", "course_id"]])
    elec = (student_electives[["student_id", "elective_id"]]
            .rename(columns={"elective_id": "course_id"}))
    return (pd.concat([core, elec])
            .sort_values(["course_id", "student_id"]).reset_index(drop=True))

registrations = build_registrations(student_electives, cohort_cores)
print(len(registrations), "rows")

_chk = registrations.merge(courses[["course_id","kind"]], on="course_id")
print("electives per student:",
      _chk[_chk.kind=="ELECTIVE"].groupby("student_id").size().unique())
registrations[registrations.student_id == "BT2S0001"]

## 6. Teachers

Related courses across years share a teacher pool — this is what makes the *invigilate your subject, not your class* rule bite.

In [ ]:
RANKS = ["PROF", "ASSOC", "ASST"]
SHARED_POOLS = [
    (["BT2_DSA"],  ["BT3_CS_SKILL"],   "CSE"),
    (["BT2_OS"],   ["BT3_COA"],        "CSE"),
    (["BT2_DLD"],  ["BT3_IOT"],        "ECE"),
    (["BT2_DAP"],  ["BT3_ML","BT3_AI"],"CSE"),
]
N_PHD_ONLY = 12

_t, _ta = [], []
_ctr = itertools.count(1)
def new_teacher(dept, rank):
    tid = f"T{next(_ctr):03d}"
    _t.append(dict(teacher_id=tid, dept=dept, rank=rank))
    return tid

def groups_of(cid):
    n = next(ns for c, _, _, ns, _ in COHORTS if c == cid)
    return [f"{cid}_{LETTERS[i]}" for i in range(n)]

def assign(pool, slots):
    k = MAX_SECTIONS_PER_TEACHER_PER_YEAR
    for i, (course, grp) in enumerate(slots):
        _ta.append((pool[(i // k) % len(pool)], course, grp))

for y2, y3, dept in SHARED_POOLS:
    s2 = [(c, g) for c in y2 for g in groups_of("BT2")]
    s3 = [(c, g) for c in y3 for g in groups_of("BT3")]
    n  = max(len(s2), len(s3)) // MAX_SECTIONS_PER_TEACHER_PER_YEAR
    pool = [new_teacher(dept, RANKS[i % len(RANKS)]) for i in range(n)]
    assign(pool, s2); assign(pool, s3)

shared = {c for a, b, _ in SHARED_POOLS for c in a + b}
for _, row in courses[courses.kind == "CORE"].iterrows():
    if row.course_id in shared: continue
    grps = groups_of(row.cohort_id)
    n = -(-len(grps) // MAX_SECTIONS_PER_TEACHER_PER_YEAR)
    pool = [new_teacher(row.dept, RANKS[i % len(RANKS)]) for i in range(n)]
    assign(pool, [(row.course_id, g) for g in grps])

for _, row in courses[courses.kind == "ELECTIVE"].iterrows():
    roll = student_electives[student_electives.elective_id == row.course_id]
    grps = [f"{row.course_id}_G{i//ELECTIVE_GROUP_SIZE + 1}" for i in range(len(roll))]
    grps = sorted(set(grps))
    n = -(-len(grps) // MAX_SECTIONS_PER_TEACHER_PER_YEAR)
    pool = [new_teacher(row.dept, RANKS[i % len(RANKS)]) for i in range(n)]
    assign(pool, [(row.course_id, g) for g in grps])

for _ in range(N_PHD_ONLY):
    new_teacher(random.choice(["CSE","ECE","MATH","MGMT"]), "PHD")

teachers = pd.DataFrame(_t)
teaching_assignments = pd.DataFrame(_ta, columns=["teacher_id","course_id","group_id"])
print(len(teachers), "teachers;", teaching_assignments.teacher_id.nunique(), "teaching")
teachers["rank"].value_counts()

## 7. Rooms, seats, stripes

A **stripe** is a set of seats that must all get the same paper. 3-wide bench -> two full-depth stripes (L/R). 2-wide bench -> two interleaved half-depth stripes (odd/even rows). Adjacent stripes must carry different papers.

In [ ]:
_r, _s = [], []
for prefix, width, per_bench, cols, rows, count in ROOM_TYPES:
    for k in range(1, count + 1):
        rid = f"{prefix}{k:02d}"
        for c in range(1, cols + 1):
            for r in range(1, rows + 1):
                if per_bench == 2:
                    for side in ("L", "R"):
                        _s.append(dict(seat_id=f"{rid}-C{c}R{r}{side}", room_id=rid,
                                       col=c, row=r, position=side,
                                       stripe_id=f"{rid}-C{c}{side}"))
                else:
                    p = "O" if r % 2 else "E"
                    _s.append(dict(seat_id=f"{rid}-C{c}R{r}", room_id=rid,
                                   col=c, row=r, position="M",
                                   stripe_id=f"{rid}-C{c}{p}"))
        _r.append(dict(room_id=rid, room_type=prefix, bench_width=width,
                       n_cols=cols, n_rows=rows, capacity=cols * rows * per_bench))

rooms = pd.DataFrame(_r)
seats = pd.DataFrame(_s)
stripes = (seats.groupby(["stripe_id","room_id"]).size()
           .reset_index(name="n_seats").sort_values("stripe_id").reset_index(drop=True))

stripe_adjacency = pd.DataFrame(
    [(a, b) for _, g in stripes.groupby("room_id")
            for a, b in itertools.combinations(sorted(g.stripe_id), 2)
            if a[:-1] == b[:-1]],
    columns=["stripe_a", "stripe_b"])

print(f"{len(rooms)} rooms, {rooms.capacity.sum()} seats, {len(stripes)} stripes")
rooms.groupby("room_type").agg(n=("room_id","size"), cap=("capacity","sum"))

## 8. Slot calendar

In [ ]:
_sl, n = [], 1
d = EXAM_START
while d <= EXAM_END:
    if not WEEKDAYS_ONLY or d.weekday() < 5:
        for sess in SESSIONS:
            _sl.append(dict(slot_id=f"SL{n:03d}", date=d.isoformat(), session=sess,
                            day_index=(d - EXAM_START).days))
            n += 1
    d += dt.timedelta(days=1)

slots = pd.DataFrame(_sl)
print(len(slots), "slots across", slots.day_index.nunique(), "days")
slots.head(4)

## 9. Write to disk

In [ ]:
TABLES = dict(students=students, sections=sections, courses=courses,
              cohort_cores=cohort_cores, student_electives=student_electives,
              registrations=registrations, teachers=teachers,
              teaching_assignments=teaching_assignments, rooms=rooms, seats=seats,
              stripes=stripes, stripe_adjacency=stripe_adjacency, slots=slots)

for name, df in TABLES.items():
    df.to_csv(f"{OUT}/{name}.csv", index=False)
for sec_id, g in students.groupby("section_id"):
    g.to_csv(f"{OUT}/sections/students_{sec_id}.csv", index=False)

pd.Series({k: len(v) for k, v in TABLES.items()}, name="rows").to_frame()

## 10. Validation

Run this after any edit. Every assert must pass.

In [ ]:
def validate():
    ok = []
    def check(name, cond, detail=""):
        ok.append((name, bool(cond), detail))

    reg = registrations.merge(courses[["course_id","kind","cohort_id"]], on="course_id")

    check("every student has exactly 1 elective",
          set(reg[reg.kind=="ELECTIVE"].groupby("student_id").size().unique()) == {1})
    check("cores complete per cohort",
          (reg[reg.kind=="CORE"].groupby(["cohort_id","student_id"]).size()
           .groupby("cohort_id").nunique() == 1).all())
    check("no duplicate registrations",
          not registrations.duplicated().any())
    check("registrations rebuild identically",
          build_registrations(student_electives, cohort_cores).equals(registrations))
    check("no orphan course_id",
          set(registrations.course_id) <= set(courses.course_id))

    ty = (teaching_assignments.merge(courses[["course_id","cohort_id"]], on="course_id")
          .groupby(["teacher_id","cohort_id"]).size().max())
    check(f"max {MAX_SECTIONS_PER_TEACHER_PER_YEAR} sections per teacher per year",
          ty <= MAX_SECTIONS_PER_TEACHER_PER_YEAR, f"observed max {ty}")

    check("seat count == room capacity",
          len(seats) == rooms.capacity.sum())
    check("all stripes non-empty", (stripes.n_seats > 0).all())

    span = slots.day_index.nunique()
    need = reg.groupby("student_id").size().max() * 2 - 1
    check("calendar long enough for 2-day gap", span >= need,
          f"{span} days available, {need} needed")

    big = registrations.groupby("course_id").size().max()
    check("largest paper <= half total seats", big <= rooms.capacity.sum() / 2,
          f"largest paper {big}, seats {rooms.capacity.sum()}")

    out = pd.DataFrame(ok, columns=["check","pass","detail"])
    assert out["pass"].all(), out[~out["pass"]]
    return out

validate()

## Where this dataset is still unrealistic

1. **No shared papers across cohorts.** `course_id` and `paper_id` are 1:1, so the
   many-codes-one-paper case is untested. Point two offerings at one `paper_id` to fix.
2. **No backlog or repeat students.** Every student's cores come purely from their cohort,
   which is exactly why the compact form works. The first repeat student breaks that
   assumption — `registrations` is the only table that can represent it.
3. **No MDC / open courses** crossing cohort boundaries.
4. **Rank pyramid is flat**, not realistic. Matters once duty fairness is rank-weighted.
5. **Zero capacity headroom** — 1200 seats, 1200 students.